In [1]:
# Run to install shap and joblib
!pip install shap; joblib; lime; xgboost

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import shap
import joblib
from lime.lime_tabular import LimeTabularExplainer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV


c:\Users\HP\anaconda3\envs\machine-learning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initialize the data

In [8]:
# Initialize and access dataset
df = pd.read_csv('data/train_set_datasheet.csv')

X = df.drop('Attrition', axis=1)
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [9]:
# Grouping by JobRoles
jobrole_dfs = {
    role: group.reset_index(drop=True)
    for role, group in df.groupby("JobRole")
}


Building an RFC model

In [10]:
# Build an RFC model
rfc = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rfc.fit(X_train, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
# Tuning the model
parameter_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", "log2"],
    "class_weight": ["balanced"]
}

grid_search = GridSearchCV(
    param_grid=parameter_grid,
    estimator=rfc,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'class_weight': ['balanced'], 'max_depth': [None, 10, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], ...}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,500


In [12]:
# Viewing the best fitting params
print("Best params: ", grid_search.best_params_)
print("Best accuracy score: ", grid_search.best_score_)

# Extracting the best model
tuned_rfc = grid_search.best_estimator_

Best params:  {'class_weight': 'balanced', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 500}
Best accuracy score:  0.9207041947402738


In [14]:
# Serializing models
joblib.dump(tuned_rfc, './models/rfc_model.joblib')
joblib.dump(X_train.columns.tolist(), './models/feature_columns.joblib')

['./models/feature_columns.joblib']

Computing SHAP values (for testing purposes)

In [3]:
# Import serialized model
serialized_model = joblib.load('./models/xgb_model.joblib')


c:\Users\HP\anaconda3\envs\machine-learning\Lib\pickle.py:1718: UserWarning: [13:46:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\gbm\../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


In [ ]:
# Inspect dtypes and the offending values
print(X_train.dtypes)
print(X_train.head())

# find any non‑numeric columns
nonnum = X_train.select_dtypes(include=['object']).columns
print("object columns:", nonnum)

# strip brackets, convert to float, coerce invalid entries to NaN
X_train[nonnum] = (
    X_train[nonnum]
      .astype(str)
      .replace(r'^\[|\]$', '', regex=True)        # '[5E-1]' → '5E-1'
      .apply(pd.to_numeric, errors='coerce')
)

# or, if you know everything should be float, simply
# X_train = X_train.replace(r'^\[|\]$', '', regex=True).astype(float)

print("after conversion:\n", X_train.dtypes.value_counts())

EmployeeID                   int64
Age                          int64
TotalWorkingYears          float64
MonthlyIncome              float64
OverTime                     int64
DistanceFromHome           float64
YearsAtCompany             float64
MonthlyRate                float64
JobInvolvement             float64
JobRole                      int64
JobLevel                   float64
YearsInCurrentRole         float64
YearsSinceLastPromotion      int64
YearsWithCurrManager         int64
EnvironmentSatisfaction    float64
Shift                      float64
WorkLifeBalance            float64
PercentSalaryHike          float64
MaritalStatus                int64
dtype: object
     EmployeeID  Age  TotalWorkingYears  MonthlyIncome  OverTime  \
832     1034814   35              0.150       0.174513         0   
266     1247928   31              0.250       0.240811         0   
148     1035340   41              0.175       0.064718         0   
383     1700683   22              0.050       0.0

In [ ]:
# Compute SHAP values
explainer = shap.TreeExplainer(
    serialized_model,
    X_train,
    feature_perturbation="interventional"
)

shap_values = explainer.shap_values(X_train)

if isinstance(shap_values, list):
    shap_attrition = shap_values[1]
else:
    shap_attrition = shap_values[:, :, 1]


mean_abs_shap_values = np.abs(shap_attrition).mean(axis=0)

print("X_train shape:", X_train.shape)
print("Number of features:", len(X_train.columns))
print("SHAP values type:", type(shap_values))
print("SHAP values shape:", np.array(shap_values).shape)

shap_importance = (
    pd.DataFrame({
        'feature': X_train.columns,
        'mean_abs_shap': mean_abs_shap_values
    }).sort_values(by='mean_abs_shap', ascending=False)
)

shap.summary_plot(shap_attrition, X_train)

In [ ]:
top_features = shap_importance["feature"].head(5).tolist()
top_features

In [ ]:
employee_idx = 0

employee_shap = shap_attrition[employee_idx]

employee_importance = pd.DataFrame({
    "feature": X_train.columns,
    "shap_value": employee_shap,
    "abs_shap": np.abs(employee_shap)
}).sort_values(by="feature", ascending=False)

print(employee_importance.head(10))

Computing LIME values (for testing purposes)

In [ ]:
# Lime 
lime_explainer = LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_train.columns.tolist(),
    class_names=['No Attrition', 'Attrition'],
    mode='classification',
)

lime_exp = lime_explainer.explain_instance(
    data_row=X_test.iloc[1].values,
    predict_fn=serialized_model.predict_proba,
    num_features=10,
    top_labels=1
)

lime_exp.as_list(label=lime_exp.available_labels()[0])

In [ ]:
# plot the lime graph
lime_exp.as_pyplot_figure(label=lime_exp.available_labels()[0]).show()

Building functions for getting results

In [7]:
def global_lime_explanation(X, model, explainer, num_features=10):
    feature_weights = defaultdict(list)

    for i in range(len(X)):
        exp = explainer.explain_instance(
            data_row=X.iloc[i].values,
            predict_fn=model.predict_proba,
            num_features=num_features,
            top_labels=1
        )

        exp_list = exp.as_list(label=exp.available_labels()[0])

        for feature, weight in exp_list:
            feature_weights[feature].append(abs(weight))

    avg_weights = {f: np.mean(w) for f, w in feature_weights.items()}
    return sorted(avg_weights.items(), key=lambda x: x[1], reverse=True)

In [ ]:
# Evaluation function
def evaluate_model(csv_path, bin_pred, pred_proba):
    results = {}

    # Load saved artifacts
    model = joblib.load('./models/xgbmodel.joblib')
    feature_columns = joblib.load('./models/feature_columns.joblib')

    #Load data
    data = pd.read_csv(csv_path)
    X_new = data[feature_columns]

    data['Predicted_Attrition_Proba'] = pred_proba
    data['Predicted_Attrition'] = bin_pred

    #Group By JobRole
    jobrole_dfs = {
        role: group.reset_index(drop=True)
        for role, group in df.groupby("JobRole")
    }

    # Lime 
    lime_explainer = LimeTabularExplainer(
        training_data=np.array(X_new),
        feature_names=X_new.columns.tolist(),
        class_names=['No Attrition', 'Attrition'],
        mode='classification',
    )

    for keys, values in jobrole_dfs.items():
        lime_result = global_lime_explanation(values, model, lime_explainer)
        results[keys] = lime_result

    return results



    

In [ ]:
def individual_values(csv_path, bin_pred, pred_proba, emp_indx):
    # Load saved artifacts
    model = joblib.load('./models/xgbmodel.joblib')
    feature_columns = joblib.load('./models/feature_columns.joblib')

    #Load data
    data = pd.read_csv(csv_path)    
    X_new = data[feature_columns]

    data['Predicted_Attrition_Proba'] = pred_proba
    data['Predicted_Attrition'] = bin_pred

    new_explainer = shap.TreeExplainer(
        tuned_rfc,
        X_train,
        feature_perturbation="interventional"
    )

    new_shap_values = explainer.shap_values(X_new)

    if isinstance(new_shap_values, list):
        new_shap_attrition = new_shap_values[1]
    else:
        new_shap_attrition = new_shap_values[:, :, 1]

    employee_idx = 0
    employee_shap = new_shap_attrition[employee_idx]

    employee_importance = pd.DataFrame({
        "feature": X_new.columns,
        "shap_value": employee_shap,
        "abs_shap": np.abs(employee_shap)
    }).sort_values(by="feature", ascending=False)

    return employee_importance.iloc[emp_indx]
